# FinOps on FOCUS — End-to-End

A complete walkthrough: build a FOCUS-conformant multi-cloud billing dataset, load it as a
semantic model, and answer the questions a FinOps practitioner actually asks.

[FOCUS](https://focus.finops.org) is the FinOps Open Cost and Usage Specification, the FinOps
Foundation's vendor-neutral schema for cost and usage data. AWS, Azure, Google Cloud and OCI all
publish FOCUS exports, so one model reads any of their bills.

FOCUS standardises the **physical** schema. It defines no metrics, no hierarchies, and since 1.4
made itself multi-dataset without defining how those datasets join. That is the gap this model fills.

**What this notebook covers**

| Step | Shows |
|---|---|
| 1-2 | Build the data, start the API |
| 3 | Cost by service (star schema) |
| 4 | **Cost allocation by tag** — reading a JSON column portably |
| 5 | **Invoice reconciliation** — a genuine multi-fact query, and what a naive join does instead |
| 6 | Rate optimisation: list vs contracted vs effective |
| 7 | Commitment utilisation, and why it warns |
| 8 | Trend and anomaly detection |


## Step 0: Build the FOCUS dataset

Six months of synthetic billing across three providers, conforming to FOCUS column names and
allowed values. The generator is committed at `scripts/build_finops_duckdb.py`; the database is
gitignored and rebuilt on demand, so this is reproducible from a clean checkout.

Five tables in a `focus` schema: `charges`, `invoice_details`, `commitments`, `providers`,
`billing_periods`.


In [ ]:
from notebook_setup import (
    create_finops_database, start_api, api,
    show_result, show_sql, show_json, show_mermaid,
)

create_finops_database()


### The raw export

The generator writes **JSONL first** and only then loads DuckDB. That ordering is deliberate: a
FOCUS export is a file you receive, not a table someone hands you. The files stay on disk under
`examples/finops_data/`, so you can open, grep and diff them before anything is modelled.

`Tags` is a **nested JSON object** in the file, exactly as a real export carries it.


In [ ]:
import json

with open("finops_data/charges.jsonl", encoding="utf-8") as fh:
    first = json.loads(fh.readline())

print(json.dumps(first, indent=2)[:1200])


The full `charges.jsonl` is about 20 MB and is gitignored. A three-record excerpt is committed at
`examples/finops_charges_sample.json` so the shape is visible without running the generator.

The load step turns that nested `Tags` object back into JSON **text**, which is what the model's
`json_value` calls read. Keeping it as a DuckDB `STRUCT` would need nested-column support that
does not exist yet.


In [ ]:
import duckdb

# fetchall rather than .df(): notebook_setup installs duckdb, pyyaml, pygments
# and sqlparse, not pandas, so a three-row preview should not pull numpy in.
con = duckdb.connect("finops.duckdb", read_only=True)
rows = con.execute("""
    SELECT ServiceName, SubAccountName, BilledCost, Tags
    FROM focus.charges WHERE Tags IS NOT NULL LIMIT 3
""").fetchall()
con.close()

for service, sub, cost, tags in rows:
    print(f"{service:<14} {sub:<16} {float(cost):>10.6f}  {tags}")


## Step 1: Start the API


In [ ]:
session_id, model_id = start_api(db_file="finops.duckdb", model_file="finops.obml.yml")
print(f"session={session_id}\nmodel={model_id}")


## Step 2: Explore the model

Two facts, three conformed dimensions. `Charges` and `Invoice Details` join to nothing but
`Billing Periods` and `Providers` — that shape is the whole reason Step 5 works.


In [ ]:
er = api("GET", "/v1/diagram/er?theme=dark")
show_mermaid(er["mermaid"])


In [ ]:
schema = api("GET", "/v1/schema")
print("dimensions:", len(schema["dimensions"]), "| measures:", len(schema["measures"]),
      "| metrics:", len(schema.get("metrics", [])))
print()
for d in schema["dimensions"][:12]:
    print(f'  {d["name"]:<18} {d.get("dataObject", "")}')


## Step 3: Cost by service

An ordinary star-schema query to get oriented.


In [ ]:
query = """
select:
  dimensions: [Provider, Service Category]
  measures: [Effective Cost, Pct of Total Spend]
orderBy:
  - field: Effective Cost
    direction: desc
limit: 8
"""
result = api("POST", "/v1/query/execute?dialect=duckdb", query)
show_result(result, query)


## Step 4: Cost allocation by tag

The first question anyone asks of a billing model: **what does each team spend?**

FOCUS answers it with a standard `Tags` column, and real exports carry it as semi-structured
data — JSON on Azure, a `MAP` on Databricks, an `ARRAY<STRUCT>` on Google Cloud. The model reads
it with `json_value`, a portable catalog function, in a computed column:

```yaml
Tags:
  code: Tags
  abstractType: json
Team Tag:
  expression: "json_value({Tags}, '$.team')"
  abstractType: string
```

`Team` is then an ordinary dimension.


In [ ]:
query = """
select:
  dimensions: [Team]
  measures: [Effective Cost, Pct of Total Spend]
orderBy:
  - field: Effective Cost
    direction: desc
"""
result = api("POST", "/v1/query/execute?dialect=duckdb", query)
show_result(result, query)


The blank row is the point. **Roughly 16% of spend carries no tags** and cannot be charged to
anyone. Untagged spend is the number a FinOps team actually chases, which is why the generator
leaves a slice of rows untagged rather than tagging everything.

### The same expression, seven dialects

`json_value` is written once. Each engine renders it into its own JSON access — and these are not
spelling variants.


In [ ]:
import re

q = """
select:
  dimensions: [Team]
  measures: [Effective Cost]
"""
# BigQuery quotes the alias with backticks, the rest with double quotes, and a
# long projection wraps, so capture from SELECT-or-comma up to the alias and
# collapse the whitespace.
PROJECTION = re.compile(r'(?:SELECT|,)\s*(.*?)\s+AS [`"]Team[`"]', re.S)

for dialect in ["bigquery", "clickhouse", "databricks", "duckdb", "postgres", "snowflake", "dremio"]:
    try:
        sql = api("POST", f"/v1/query/sql?dialect={dialect}", q)["sql"]
    except Exception:
        print(f"  {dialect:<11} unsupported: no JSONPath scalar function")
        continue
    m = PROJECTION.search(sql)
    print(f'  {dialect:<11} {re.sub(r"\s+", " ", m.group(1)) if m else "?"}')


Four things differ there, and each was measured against a live engine rather than assumed:

- **Postgres** takes the path segments as *separate arguments*; **Snowflake** takes them dotted
  without the `$`, and bracketed for array subscripts — it rejects `arr.0` outright.
- **ClickHouse** returns the *empty string* rather than NULL for an absent path, so it is wrapped
  in `nullIf`.
- **DuckDB, Postgres, Snowflake and MySQL** return the *serialized JSON* when the path lands on an
  object or array, so each carries a type guard to honour the catalog's NULL rule. That is where
  the `CASE` comes from.
- **Databricks** is the only engine that gets the contract for free: `try_variant_get(…, 'string')`
  answers NULL when the value will not cast, which is the object/array rule and the absent-path
  rule at once.

That spread is why the path must be a literal, and why `json_value` belongs in the catalog rather
than being hand-written per model.

Dremio is the exception: no JSONPath scalar function, so it reports the call unsupported at compile
time — a real HTTP 422 from the API above — rather than mis-rendering it.


## Step 5: Invoice reconciliation

**Does what we were charged match what we were invoiced?** FOCUS 1.4's headline use case.

`Billed Cost` lives on `Charges`; `Invoiced Amount` lives on `Invoice Details`. Nothing joins
those two objects — they meet only at the conformed dimensions. So this is a genuine multi-fact
query, and OrionBelt answers it through the composite fact layer.


In [ ]:
query = """
select:
  dimensions: [Billing Period, Provider]
  measures: [Billed Cost, Invoiced Amount, Invoice Variance]
orderBy:
  - field: Billing Period
    direction: asc
limit: 6
"""
result = api("POST", "/v1/query/execute?dialect=duckdb", query)
show_result(result, query)


The result above carries the compiled SQL with it - `show_result` renders the OBML query, the SQL
and the table together. Look at the SQL panel: the two facts are stacked with `UNION ALL` and the
missing columns NULL-padded, so each is aggregated at its own grain rather than joined.

### What the obvious alternative does

Joining both facts to the billing period in one query is the intuitive move, and it is wrong.


In [ ]:
import duckdb

con = duckdb.connect("finops.duckdb", read_only=True)
truth = con.execute(
    "SELECT (SELECT SUM(BilledCost) FROM focus.charges),"
    "       (SELECT SUM(InvoicedAmount) FROM focus.invoice_details)"
).fetchone()
naive = con.execute("""
    SELECT SUM(c.BilledCost), SUM(i.InvoicedAmount)
    FROM focus.charges c
    JOIN focus.invoice_details i
      ON c.BillingPeriodStart = i.BillingPeriodStart
     AND c.ProviderName = i.ProviderName
""").fetchone()
con.close()

print(f"{'':<26}{'Billed Cost':>18}{'Invoiced Amount':>20}")
print(f"{'truth':<26}{float(truth[0]):>18,.2f}{float(truth[1]):>20,.2f}")
print(f"{'charges JOIN invoices':<26}{float(naive[0]):>18,.2f}{float(naive[1]):>20,.2f}")
print(f"\ninvoiced amount inflated {float(naive[1]) / float(truth[1]):,.0f}x")


Every charge row is multiplied by every invoice line for its provider and period. Nothing errors;
the dashboard just reports a number three orders of magnitude wrong.


## Step 6: Rate optimisation

FOCUS separates three costs, and the gaps between them are where the savings live:

| Measure | Meaning |
|---|---|
| `List Cost` | at published on-demand rates |
| `Contracted Cost` | at negotiated rates |
| `Effective Cost` | with commitments amortised |

Splitting the two rates answers what a single savings number cannot: how much came from
negotiating, and how much from committing.


In [ ]:
query = """
select:
  dimensions: [Provider, Service Category]
  measures: [Effective Cost, List Cost, Effective Savings Rate, Negotiated Discount Rate]
orderBy:
  - field: Effective Cost
    direction: desc
limit: 6
"""
result = api("POST", "/v1/query/execute?dialect=duckdb&format_values=true", query)
show_result(result, query)


## Step 7: Commitment utilisation

How much of the committed spend did we actually draw? Under 100% means money is going to waste.


In [ ]:
query = """
select:
  dimensions: [Commitment, Commitment Type]
  measures: [Effective Cost, Committed Amount, Commitment Utilization]
"""
result = api("POST", "/v1/query/execute?dialect=duckdb&format_values=true", query)
show_result(result, query)
for w in result.get("warnings", []):
    print("warning:", w if isinstance(w, str) else w.get("message"))


That warning is correct behaviour, not a defect. `Committed Amount` lives on the *one* side of a
many-to-one join: one commitment covers thousands of charge rows. Summing it naively across the
joined result would multiply the contract value by the number of charges it covered. OrionBelt
deduplicates on the commitment key so each group is right, and tells you the column will not
cross-foot.


## Step 8: Trend and anomaly detection

Cumulative and period-over-period metrics from the same model. `Effective Cost MoM Growth` is
what a cost-anomaly alert fires on.


In [ ]:
query = """
select:
  dimensions: [Charge Date]
  measures: [Effective Cost, Running Effective Cost, Effective Cost MoM Growth]
"""
result = api("POST", "/v1/query/execute?dialect=duckdb&format_values=true", query)
show_result(result, query)


## Next steps

- Point it at a **real export**: change `code` and `schema` on the data objects and set
  `settings.defaultDialect`. Every column modelled here is a standard FOCUS column.
- Google Cloud carries labels, credits and tags as `REPEATED RECORD` extension columns prefixed
  `x_`. Reach those through a flattening view for now. Note that `x_Credits` being repeated means
  credits are a **second fact at a different grain**, not a labels convenience.
- The model sets `expressionMode: portable`, so an uncatalogued function is an error rather than a
  silent engine dependency.

See [FinOps on FOCUS](https://ralforion.com/orionbelt-semantic-layer/examples/finops-focus/) for
the written walkthrough.
